# Week 19, Day 5: Variational inference: approximating the posterior

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w019/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Check the bound

Prior z ~ N(0, 1), one observation x = 1.2 with x | z ~ N(z, 0.5²). Write `elbo(m, s, rng)` estimating E_q[log p(x, z)] − E_q[log q(z)] with 100,000 samples from q = N(m, s²). Compute the exact posterior (`post_m`, `post_s`, Week 16's precision rule) and the exact `log_evidence` (x ~ N(0, 1 + 0.5²)), then evaluate the ELBO for a few choices of q.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2017)
x_obs, s_lik = 1.2, 0.5
def log_norm(v, m, s): return -0.5 * np.log(2 * np.pi * s ** 2) - (v - m) ** 2 / (2 * s ** 2)

def elbo(m, s, rng):
    z = m + s * rng.normal(size=100000)
    log_joint = ...                          # log p(x | z) + log p(z)
    log_q = ...
    return np.mean(log_joint - log_q)

prec = 1 + 1 / s_lik ** 2
post_m, post_s = (x_obs / s_lik ** 2) / prec, 1 / np.sqrt(prec)
log_evidence = log_norm(x_obs, 0, np.sqrt(1 + s_lik ** 2))
qs = [(0.0, 1.0), (1.2, 0.5), (post_m, post_s), (0.96, 0.2)]
elbos = np.array([elbo(m, s, rng) for m, s in qs])
print(f"log evidence {log_evidence:.4f}");  [print(f"q = N({m:.2f}, {s:.2f}^2): ELBO {e:.4f}") for (m, s), e in zip(qs, elbos)]

In [ ]:
import numpy as np
def test_bound():
    "the ELBO is below the log evidence for every q tried, by more than 0.01 for a poor q"
    assert np.all(elbos <= log_evidence + 0.01) and elbos.min() < log_evidence - 0.01

def test_tight():
    "at the exact posterior the ELBO equals the log evidence to within 0.01"
    assert abs(elbo(post_m, post_s, np.random.default_rng(1)) - log_evidence) < 0.01

test_bound()
test_tight()
print("All checks passed")

In [ ]:
r_v = np.random.default_rng(60)
theta = r_v.uniform(0, np.pi, 60)
X = np.column_stack([np.ones(60), np.cos(2 * theta), np.sin(2 * theta)])
k = r_v.poisson(np.exp(1 + 0.8 * np.cos(2 * (theta - np.deg2rad(60)))))
def log_post(w): eta = X @ w; return np.sum(k * eta - np.exp(eta)) - np.sum(w ** 2) / 18
def grad_log_post(W):                     # W has one weight vector per row
    return (k - np.exp(W @ X.T)) @ X - W / 9

# A long Metropolis run, as the reference answer
w, lp, chain = np.zeros(3), log_post(np.zeros(3)), []
for t in range(40000):
    prop = w + 0.12 * r_v.normal(size=3); lpp = log_post(prop)
    if np.log(r_v.random()) < lpp - lp: w, lp = prop, lpp
    chain.append(w.copy())
chain = np.array(chain[4000:])
print("MCMC posterior means", chain.mean(axis=0).round(3), " sds", chain.std(axis=0).round(3))

### Exercise: Fit the variational Gaussian

Run 3,000 steps of Adam (step size 0.02) on the ELBO, starting from m = 0 and s = 0.3, with 20 samples of ε per step. Store the fitted means in `m` and standard deviations in `s_vi`, and compare with MCMC.

In [ ]:
m, log_s = np.zeros(3), np.log(0.3) * np.ones(3)
am, av = np.zeros(6), np.zeros(6)
for it in range(1, 3001):
    eps = r_v.normal(size=(20, 3))
    s = np.exp(log_s)
    G = grad_log_post(m + s * eps)                  # 20 x 3
    g_m = ...
    g_log_s = ...
    grad = np.r_[g_m, g_log_s]
    am = 0.9 * am + 0.1 * grad; av = 0.999 * av + 0.001 * grad ** 2
    step = 0.02 * (am / (1 - 0.9 ** it)) / (np.sqrt(av / (1 - 0.999 ** it)) + 1e-8)
    m, log_s = m + step[:3], log_s + step[3:]       # ascent: we maximise the ELBO
s_vi = np.exp(log_s)
print("variational means", m.round(3), " sds", s_vi.round(3))
print("MCMC        means", chain.mean(axis=0).round(3), " sds", chain.std(axis=0).round(3))

In [ ]:
import numpy as np
def test_means():
    "the variational means agree with the MCMC means to within 0.06"
    assert np.all(np.abs(m - chain.mean(axis=0)) < 0.06)

def test_sds():
    "the variational sds are no larger than the MCMC sds (within 5%), and at least one is more than 10% smaller"
    s_mc = chain.std(axis=0)
    assert np.all(s_vi <= 1.05 * s_mc) and np.any(s_vi < 0.9 * s_mc)

test_means()
test_sds()
print("All checks passed")